# Integración de IGEIP e INCO 2024 con Dataset de Obras Públicas (`ds_op`)

Este notebook implementa el cruce del dataset base de obras públicas (`r$_ds_obras_publicas_infobras`) con:
1. **INCO 2024 (Índice de Riesgos de la Corrupción e Inconducta Funcional)** a nivel de Entidad (`codigo_entidad` = `COD ENTIDAD`).
2. **IGEIP (Índice de Gestión Estratégica de la Inversión Pública)** a nivel Sectorial para entidades de `GOBIERNO NACIONAL` aplicando matching semántico y expresiones regulares.
3. **Validaciones de integridad** de los datos cruzados.
4. **Reporte ejecutivo de resultados** de cobertura y correspondencia.

## 1. Conexión a la base de datos y extracción del dataset base (`ds_op`)

In [1]:
import sys
from pathlib import Path
from configparser import ConfigParser
import pandas as pd
import numpy as np
import re
import unicodedata

# Detección robusta de la raíz del proyecto
project_root = Path.cwd() if (Path.cwd() / 'configs').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

from db.connection import PostgresConnection

# 1. Cadena de conexión provista
config_path = project_root / 'configs' / 'database.ini'
config = ConfigParser()
config.read(config_path)

pg_conn = PostgresConnection(
    host=config.get('postgres', 'host'),
    port=config.getint('postgres', 'port'),
    dbname=config.get('postgres', 'dbname'),
    user=config.get('postgres', 'user'),
    password=config.get('postgres', 'password'),
)
conn = pg_conn.connect()
print(f'Conexion establecida: {pg_conn.host}:{pg_conn.port}/{pg_conn.dbname}')

# 2. Carga de la tabla base ds_op
table_name = 'r$_ds_obras_publicas_infobras'
print(f"Cargando tabla base '{table_name}'...")
query = f'SELECT * FROM "{table_name}"'
ds_op = pd.read_sql(query, conn)
print(f"Dataset base 'ds_op' cargado: {len(ds_op):,} registros, {len(ds_op.columns)} columnas")

Conexion establecida: localhost:5432/postgres
Cargando tabla base 'r$_ds_obras_publicas_infobras'...


C:\Users\yanfr\AppData\Local\Temp\ipykernel_49712\1204665027.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ds_op = pd.read_sql(query, conn)


Dataset base 'ds_op' cargado: 69,952 registros, 130 columnas


## 2. Preparación y carga de INCO 2024
Regla 3: Cruce left join `ds_op.codigo_entidad = df_inco."COD ENTIDAD"`.
Regla 5: Integrar `df_inco."PUNTAJE INCO 2024"` as `inco_2024_entidad`.

In [2]:
# Carga del archivo INCO 2024
inco_path = project_root / 'data' / 'raw' / 'inco' / 'inco-2024.xlsx'
df_inco = pd.read_excel(inco_path, sheet_name='inco-2024')
print(f'Dimensiones df_inco original: {df_inco.shape}')

# Preparar subconjunto y renombrar campo solicitado
df_inco_subset = df_inco[['COD ENTIDAD', 'PUNTAJE INCO 2024']].copy()
df_inco_subset.rename(columns={
    'COD ENTIDAD': 'cod_entidad_inco',
    'PUNTAJE INCO 2024': 'inco_2024_entidad'
}, inplace=True)

# Limpiar y asegurar claves textuales sin espacios
df_inco_subset['cod_entidad_inco'] = df_inco_subset['cod_entidad_inco'].astype(str).str.strip()
ds_op['codigo_entidad_str'] = ds_op['codigo_entidad'].astype(str).str.strip()

# Cruce Left Join con INCO 2024
ds_op_inco = ds_op.merge(
    df_inco_subset,
    left_on='codigo_entidad_str',
    right_on='cod_entidad_inco',
    how='left'
).drop(columns=['cod_entidad_inco', 'codigo_entidad_str'])

print(f'Cruce INCO 2024 realizado. Dimensiones: {ds_op_inco.shape}')
print(f'Obras con puntaje INCO asignado: {ds_op_inco["inco_2024_entidad"].notna().sum():,} / {len(ds_op_inco):,}')

Dimensiones df_inco original: (2926, 12)
Cruce INCO 2024 realizado. Dimensiones: (69952, 131)
Obras con puntaje INCO asignado: 69,640 / 69,952


## 3. Preparación de IGEIP y Mapeo Semántico/Regex por Sector
Regla 4: Cruce entre `ds_op.sector_entidad` y `df_igeip.sector` usando match semántico / expresiones regulares.
El cruce aplica exclusivamente para cuando `ds_op.nivel_gobierno == 'GOBIERNO NACIONAL'`.
Regla 5: Integrar `df_igeip.igeip` as `igeip_sector.*` (incluyendo dimensiones e índice general).

In [3]:
# Estructura de datos IGEIP por Sector
datos_igeip = [
    {'sector': 'ECONOMÍA Y FINANZAS', 'programacion_multianual': 0.628, 'formulacion_evaluacion': 0.725, 'ejecucion': 0.673, 'funcionamiento': 0.800, 'gobernabilidad': 0.980, 'igeip': 0.739},
    {'sector': 'PCM', 'programacion_multianual': 0.629, 'formulacion_evaluacion': 0.746, 'ejecucion': 0.702, 'funcionamiento': np.nan, 'gobernabilidad': 0.864, 'igeip': 0.729},
    {'sector': 'CULTURA', 'programacion_multianual': 0.630, 'formulacion_evaluacion': 0.776, 'ejecucion': 0.690, 'funcionamiento': np.nan, 'gobernabilidad': 0.842, 'igeip': 0.726},
    {'sector': 'ENERGIA Y MINAS', 'programacion_multianual': 0.631, 'formulacion_evaluacion': 0.782, 'ejecucion': 0.693, 'funcionamiento': np.nan, 'gobernabilidad': 0.714, 'igeip': 0.702},
    {'sector': 'COMERCIO EXTERIOR Y TURISMO', 'programacion_multianual': 0.700, 'formulacion_evaluacion': 0.760, 'ejecucion': 0.625, 'funcionamiento': np.nan, 'gobernabilidad': 0.789, 'igeip': 0.700},
    {'sector': 'TRANSPORTES Y COMUNICACIONES', 'programacion_multianual': 0.701, 'formulacion_evaluacion': 0.727, 'ejecucion': 0.573, 'funcionamiento': np.nan, 'gobernabilidad': 0.841, 'igeip': 0.683},
    {'sector': 'SALUD', 'programacion_multianual': 0.586, 'formulacion_evaluacion': 0.795, 'ejecucion': 0.542, 'funcionamiento': 0.857, 'gobernabilidad': 0.856, 'igeip': 0.674},
    {'sector': 'TRABAJO Y PROMOCION DEL EMPLEO', 'programacion_multianual': 0.227, 'formulacion_evaluacion': 0.708, 'ejecucion': 0.772, 'funcionamiento': np.nan, 'gobernabilidad': 0.888, 'igeip': 0.673},
    {'sector': 'MUJER Y POBLACIONES VULNERABLES', 'programacion_multianual': 0.523, 'formulacion_evaluacion': 0.847, 'ejecucion': 0.492, 'funcionamiento': np.nan, 'gobernabilidad': 0.978, 'igeip': 0.666},
    {'sector': 'DEFENSA', 'programacion_multianual': 0.679, 'formulacion_evaluacion': 0.748, 'ejecucion': 0.543, 'funcionamiento': np.nan, 'gobernabilidad': 0.784, 'igeip': 0.659},
    {'sector': 'EDUCACIÓN', 'programacion_multianual': 0.606, 'formulacion_evaluacion': 0.661, 'ejecucion': 0.626, 'funcionamiento': np.nan, 'gobernabilidad': 0.778, 'igeip': 0.659},
    {'sector': 'AMBIENTAL', 'programacion_multianual': 0.576, 'formulacion_evaluacion': 0.679, 'ejecucion': 0.587, 'funcionamiento': np.nan, 'gobernabilidad': 0.857, 'igeip': 0.657},
    {'sector': 'AGRICULTURA Y RIEGO', 'programacion_multianual': 0.530, 'formulacion_evaluacion': 0.766, 'ejecucion': 0.586, 'funcionamiento': np.nan, 'gobernabilidad': 0.814, 'igeip': 0.656},
    {'sector': 'PRODUCCION', 'programacion_multianual': 0.581, 'formulacion_evaluacion': 0.699, 'ejecucion': 0.534, 'funcionamiento': np.nan, 'gobernabilidad': 0.870, 'igeip': 0.644},
    {'sector': 'JUSTICIA', 'programacion_multianual': 0.499, 'formulacion_evaluacion': 0.686, 'ejecucion': 0.587, 'funcionamiento': np.nan, 'gobernabilidad': 0.778, 'igeip': 0.627},
    {'sector': 'INTERIOR', 'programacion_multianual': 0.436, 'formulacion_evaluacion': 0.666, 'ejecucion': 0.559, 'funcionamiento': 0.000, 'gobernabilidad': 0.825, 'igeip': 0.609},
    {'sector': 'VIVIENDA, CONSTRUCCIÓN Y SANEAMIENTO', 'programacion_multianual': 0.549, 'formulacion_evaluacion': 0.665, 'ejecucion': 0.560, 'funcionamiento': np.nan, 'gobernabilidad': 0.690, 'igeip': 0.605},
    {'sector': 'DESARROLLO E INCLUSIÓN SOCIAL', 'programacion_multianual': 0.420, 'formulacion_evaluacion': 0.688, 'ejecucion': 0.415, 'funcionamiento': np.nan, 'gobernabilidad': 0.638, 'igeip': 0.515},
    {'sector': 'RELACIONES EXTERIORES', 'programacion_multianual': 0.323, 'formulacion_evaluacion': 0.550, 'ejecucion': 0.400, 'funcionamiento': np.nan, 'gobernabilidad': 0.411, 'igeip': 0.417},
]
df_igeip = pd.DataFrame(datos_igeip)

# Renombrar columnas con prefijo igeip_sector.* (Regla 5)
df_igeip_renamed = df_igeip.rename(columns={
    'igeip': 'igeip_sector',
    'programacion_multianual': 'igeip_sector_programacion_multianual',
    'formulacion_evaluacion': 'igeip_sector_formulacion_evaluacion',
    'ejecucion': 'igeip_sector_ejecucion',
    'funcionamiento': 'igeip_sector_funcionamiento',
    'gobernabilidad': 'igeip_sector_gobernabilidad'
})
df_igeip_renamed.head(3)

,sector,igeip_sector_programacion_multianual,igeip_sector_formulacion_evaluacion,igeip_sector_ejecucion,igeip_sector_funcionamiento,igeip_sector_gobernabilidad,igeip_sector
0,ECONOMÍA Y FINANZAS,0.628,0.725,0.673,0.8,0.980,0.739
1,PCM,0.629,0.746,0.702,NaN,0.864,0.729
2,CULTURA,0.630,0.776,0.690,NaN,0.842,0.726


In [4]:
# Normalización de texto y mapeo semántico / regex
def normalizar_cadena(s: str) -> str:
    if not isinstance(s, str):
        return ''
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip().upper()

# Reglas semánticas por expresiones regulares
REGLAS_SEMANTICAS_SECTOR = [
    (r'\b(PCM|PRESIDENCIA CONSEJO MINISTROS)\b', 'PCM'),
    (r'\bAGRICULTURA\b|\bAGRARIO\b', 'AGRICULTURA Y RIEGO'),
    (r'\bVIVIENDA\b.*\bCONSTRUCCION\b.*\bSANEAMIENTO\b', 'VIVIENDA, CONSTRUCCIÓN Y SANEAMIENTO'),
    (r'\bTRABAJO\b.*\bEMPLEO\b', 'TRABAJO Y PROMOCION DEL EMPLEO'),
    (r'\bDESARROLLO\b.*\bINCLUSION\b', 'DESARROLLO E INCLUSIÓN SOCIAL'),
    (r'\bCOMERCIO\b.*\bTURISMO\b', 'COMERCIO EXTERIOR Y TURISMO'),
    (r'\bTRANSPORTES\b.*\bCOMUNICACIONES\b', 'TRANSPORTES Y COMUNICACIONES'),
    (r'\bMUJER\b.*\bVULNERABLES\b', 'MUJER Y POBLACIONES VULNERABLES'),
    (r'\bENERGIA\b.*\bMINAS\b', 'ENERGIA Y MINAS'),
]

def resolver_sector_igeip(row) -> str | None:
    # Regla 4: Cruce solo para cuando nivel_gobierno == 'GOBIERNO NACIONAL'
    if row.get('nivel_gobierno') != 'GOBIERNO NACIONAL':
        return None
    sec = row.get('sector_entidad')
    if not isinstance(sec, str) or not sec.strip():
        return None
    sec_norm = normalizar_cadena(sec)
    for patron, target in REGLAS_SEMANTICAS_SECTOR:
        if re.search(patron, sec_norm):
            return target
    igeip_norm_map = {normalizar_cadena(s): s for s in df_igeip['sector']}
    return igeip_norm_map.get(sec_norm, None)

# Asignar clave temporal para merge
ds_op_inco['_sector_igeip_clave'] = ds_op_inco.apply(resolver_sector_igeip, axis=1)

# Cruce Left Join con IGEIP
ds_op_final = ds_op_inco.merge(
    df_igeip_renamed,
    left_on='_sector_igeip_clave',
    right_on='sector',
    how='left'
).drop(columns=['_sector_igeip_clave', 'sector'])

print(f'Cruce IGEIP completado.')
print(f'Dimensiones del dataset integrado ds_op_final: {ds_op_final.shape}')

Cruce IGEIP completado.
Dimensiones del dataset integrado ds_op_final: (69952, 137)


## 4. Scripts de Validación de Integridad de los Cruces

In [5]:
print('=' * 50)
print('     VALIDACIONES DE INTEGRIDAD DEL DATASET       ')
print('=' * 50)

# 1. Validación de número de filas (sin duplicaciones no deseadas)
assert len(ds_op_final) == len(ds_op), (
    f'ERROR: Discrepancia en filas. Original: {len(ds_op)}, Final: {len(ds_op_final)}'
)
print(f'[OK] Validación 1: Filas consistentes ({len(ds_op_final):,} registros).')

# 2. Validación de presencia de columnas requeridas
columnas_requeridas = [
    'inco_2024_entidad',
    'igeip_sector',
    'igeip_sector_programacion_multianual',
    'igeip_sector_formulacion_evaluacion',
    'igeip_sector_ejecucion',
    'igeip_sector_funcionamiento',
    'igeip_sector_gobernabilidad'
]
for col in columnas_requeridas:
    assert col in ds_op_final.columns, f'ERROR: Columna faltante {col}'
print(f'[OK] Validación 2: Todas las {len(columnas_requeridas)} nuevas columnas existen en ds_op_final.')

# 3. Validación de restricción de GOBIERNO NACIONAL para IGEIP (Regla 4)
obras_no_gn_con_igeip = ds_op_final.loc[
    ds_op_final['nivel_gobierno'] != 'GOBIERNO NACIONAL', 'igeip_sector'
].notna().sum()
assert obras_no_gn_con_igeip == 0, (
    f'ERROR: Hay {obras_no_gn_con_igeip} obras que no son GOBIERNO NACIONAL con IGEIP.'
)
print('[OK] Validación 3: Restricción cumplida (IGEIP solo asignado a GOBIERNO NACIONAL).')

# 4. Validación de coherencia de rangos numéricos
assert ds_op_final['inco_2024_entidad'].dropna().between(0, 100).all(), 'Puntaje INCO fuera de rango [0, 100]'
assert ds_op_final['igeip_sector'].dropna().between(0, 1).all(), 'Índice IGEIP fuera de rango [0, 1]'
print('[OK] Validación 4: Rangos de puntajes válidos (INCO en [0, 100], IGEIP en [0, 1]).')
print('=' * 50)
print('TODAS LAS VALIDACIONES PASARON SATISFACTORIAMENTE.')

     VALIDACIONES DE INTEGRIDAD DEL DATASET       
[OK] Validación 1: Filas consistentes (69,952 registros).
[OK] Validación 2: Todas las 7 nuevas columnas existen en ds_op_final.
[OK] Validación 3: Restricción cumplida (IGEIP solo asignado a GOBIERNO NACIONAL).
[OK] Validación 4: Rangos de puntajes válidos (INCO en [0, 100], IGEIP en [0, 1]).
TODAS LAS VALIDACIONES PASARON SATISFACTORIAMENTE.


## 5. Reporte Final de Resultados de los Cruces

In [6]:
total_obras = len(ds_op_final)

# Métricas Cruce 1: INCO 2024
inco_match = ds_op_final['inco_2024_entidad'].notna().sum()
inco_no_match = total_obras - inco_match

# Métricas Cruce 2: IGEIP Sectorial (Gobierno Nacional)
mask_gn = ds_op_final['nivel_gobierno'] == 'GOBIERNO NACIONAL'
total_gn = mask_gn.sum()
igeip_gn_match = (mask_gn & ds_op_final['igeip_sector'].notna()).sum()
igeip_gn_no_match = total_gn - igeip_gn_match

reporte_resumen = pd.DataFrame([
    {
        'Cruce': '1. INCO 2024 (por Entidad)',
        'Población Evaluada': f'{total_obras:,} obras (Total)',
        'Con Match': f'{inco_match:,}',
        '% Cobertura': f'{inco_match / total_obras:.2%}',
        'Sin Match': f'{inco_no_match:,}',
        '% Sin Match': f'{inco_no_match / total_obras:.2%}'
    },
    {
        'Cruce': '2. IGEIP (por Sector)',
        'Población Evaluada': f'{total_gn:,} obras (Gob. Nacional)',
        'Con Match': f'{igeip_gn_match:,}',
        '% Cobertura': f'{igeip_gn_match / total_gn:.2%}',
        'Sin Match': f'{igeip_gn_no_match:,}',
        '% Sin Match': f'{igeip_gn_no_match / total_gn:.2%}'
    }
])

print('=== REPORTE EJECUTIVO DE CRUCES ===')
print(reporte_resumen.to_string(index=False))

# Detalle por Sector de Gobierno Nacional
print('\n=== DETALLE DE COBERTURA IGEIP POR SECTOR (GOBIERNO NACIONAL) ===')
detalle_gn = ds_op_final[mask_gn].groupby('sector_entidad').agg(
    total_obras=('codigo_infobras', 'count'),
    con_igeip=('igeip_sector', lambda s: s.notna().sum()),
    igeip_asignado=('igeip_sector', 'first')
).reset_index()

detalle_gn['pct_cobertura'] = (detalle_gn['con_igeip'] / detalle_gn['total_obras'] * 100).map('{:.1f}%'.format)
detalle_gn.sort_values(by='total_obras', ascending=False, inplace=True)
print(detalle_gn.to_string(index=False))

=== REPORTE EJECUTIVO DE CRUCES ===
                     Cruce          Población Evaluada Con Match % Cobertura Sin Match % Sin Match
1. INCO 2024 (por Entidad)        69,952 obras (Total)    69,640      99.55%       312       0.45%
     2. IGEIP (por Sector) 7,367 obras (Gob. Nacional)     6,461      87.70%       906      12.30%

=== DETALLE DE COBERTURA IGEIP POR SECTOR (GOBIERNO NACIONAL) ===
                     sector_entidad  total_obras  con_igeip  igeip_asignado pct_cobertura
                          EDUCACIÓN         1752       1752           0.659        100.0%
VIVIENDA CONSTRUCCIÓN Y SANEAMIENTO          940        940           0.605        100.0%
       TRANSPORTES Y COMUNICACIONES          822        822           0.683        100.0%
                        AGRICULTURA          816        816           0.656        100.0%
                             FONAFE          764          0             NaN          0.0%
                    ENERGÍA Y MINAS          479        479 

## 6. Guardado del Dataset Integrado en Capa Gold
Exportación del dataset procesado `ds_op_final` a formato CSV en `data/gold/ds_op_inco_igeip.csv`.

In [10]:
# Definir directorio de destino en data/gold
output_dir = project_root / 'data' / 'gold'
output_dir.mkdir(parents=True, exist_ok=True)


# Guardar dataset integrado en formato CSV
output_file = output_dir / 'ds_op_inco_igeip.csv'
ds_op_final.to_csv(output_file, index=False, encoding='utf-8')

print(f'[OK] Dataset integrado guardado exitosamente en: {output_file}')
print(f'Dimensiones del archivo guardado: {ds_op_final.shape}')
print(f'Tamaño del archivo generado: {output_file.stat().st_size / (1024 * 1024):.2f} MB')

[OK] Dataset integrado guardado exitosamente en: c:\ws\formacion\uni-mia\mia-tesis-opa\data\gold\ds_op_inco_igeip.csv
Dimensiones del archivo guardado: (69952, 137)
Tamaño del archivo generado: 94.64 MB
